In [ ]:
# install packages required

%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers
%pip install datasets
%pip install scikit-learn
%pip install 'accelerate>=1.1.0'
%pip install time
%pip install evaluate

In [7]:
# loading packages

import duckdb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import TrainingArguments, Trainer
import transformers
import accelerate
from transformers import BertForSequenceClassification
import time
import torch
import numpy as np
import evaluate
from transformers import BertTokenizer, BertForSequenceClassification
import pickle
import os
import s3fs


In [8]:
# loading the dataset

con = duckdb.connect(database=":memory:")
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

query = "SELECT * FROM read_parquet('https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet')"
df = con.sql(query).df()

In [9]:
df.info()

In [10]:
#import the NACE

con = duckdb.connect(database=":memory:")

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

path_nace = 'https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/NACE_Rev2.1_Structure_Explanatory_Notes_EN.tsv'
query_definition = f"SELECT * FROM read_csv('{path_nace}')"
table = con.execute(query_definition).to_arrow_table()
nace = table.to_pylist()
nace[1]

nace2 = pd.DataFrame(nace)


In [ ]:
nace2.head()

saving the NACE code for future use

In [34]:
nace_filtered = nace2[nace2["CODE"].astype(str).str.len() == 5]
nace_filtered= nace_filtered[['CODE', 'HEADING','Includes', 'IncludesAlso', 'Excludes' ]]
nace_filtered["Includes"] = nace_filtered["Includes"].str.replace("\\n", "", regex=False)
nace_filtered["IncludesAlso"] = nace_filtered["IncludesAlso"].str.replace("\\n", "", regex=False)
nace_filtered["Excludes"] = nace_filtered["Excludes"].str.replace("\\n", "", regex=False)
nace_filtered.head()
#nomenclature.head()
df.head()


#save test_df for the nexte notebook
nace_filtered.to_pickle("nace.pkl")

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "thierry57"
FILE_KEY_OUT_S3 = "model_nace/nace.pkl"
FILE_PATH_OUT_S3 = BUCKET_OUT + "/" + FILE_KEY_OUT_S3


with fs.open(FILE_PATH_OUT_S3, 'wb') as file_out:
    nace_filtered.to_pickle(file_out)

Encoding NACE codes using LabelEncoder

In [13]:
# label encoding
le = LabelEncoder()
df['target'] = le.fit_transform(df['code'])

num_labels = df['target'].nunique()
print(num_labels)

I create the training and test datasets by adding a stratification parameter, as the classes are highly imbalanced

In [30]:
# create train and test dataset
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['target'],  # class equilibtrate
    random_state=42
)


#save test_df for the next notebook
test_df.to_pickle("test_df.pkl")

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "thierry57"
FILE_KEY_OUT_S3 = "model_nace/test_df"
FILE_PATH_OUT_S3 = BUCKET_OUT + "/" + FILE_KEY_OUT_S3


with fs.open(FILE_PATH_OUT_S3, 'wb') as file_out:
    test_df.to_pickle(file_out)

loads a tokeniser from the BERT model, which converts the text into a numerical vector.


In [31]:
# tokenize

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(
        batch['label'],
        padding='max_length',
        truncation=True,
        max_length=128
    )


Converts training and test data into a format compatible with the Hugging Face library and PyTorch.
It then applies the tokenisation function to the text, renames the target column, and prepares the final data so that it can be used directly by the model.

In [ ]:

# conversion in datasets
train_dataset = Dataset.from_pandas(train_df[['label', 'target']])
test_dataset = Dataset.from_pandas(test_df[['label', 'target']])

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("target", "labels")
test_dataset = test_dataset.rename_column("target", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


# temporary dataset reduced for test the code
#train_dataset = train_dataset.shuffle().select(range(20))
#test_dataset = test_dataset.shuffle().select(range(4))


Next, I load the Bert model using a case-insensitive version, as I don’t think it’s necessary here

In [ ]:

# charge bert model

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

Next, I start fine-tuning the model, limiting the number of epochs to 3 to avoid overfitting

With an SSPCloud GPU, it takes about 2 hours

Then I immediately save the model and the encoder, as I’m using a GPU and need to free up resources quickly

I save it locally first, then to S3 storage

In [ ]:
# trainning

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_dir="./logs",
    disable_tqdm=False,   
    report_to="none"     
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

start = time.time()

trainer.train()

end = time.time()

print(f"Training time : {end - start:.2f} secondes")

### saving model, tokenizer, encoder and nace
# dossier local temporaire
LOCAL_DIR = "./model_nace"
os.makedirs(LOCAL_DIR, exist_ok=True)

# save local
trainer.save_model("./model_nace")
tokenizer.save_pretrained("./model_nace")
with open("./model_nace/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

nomenclature.to_csv("./model_nace/nomenclature.csv", index=False)


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

trainer.save_model(LOCAL_DIR)
tokenizer.save_pretrained(LOCAL_DIR)

with open(f"{LOCAL_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

nomenclature.to_csv(f"{LOCAL_DIR}/nomenclature.csv", index=False)

# -------------------
# upload to bucket
# -------------------

BUCKET_OUT = "thierry57"

files_to_upload = [
    "config.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "model.safetensors",        
    "label_encoder.pkl",
    "nomenclature.csv",
    "training_args.bin"
]

for file_name in files_to_upload:

    local_path = f"{LOCAL_DIR}/{file_name}"
    s3_path = f"{BUCKET_OUT}/model_nace/{file_name}"

    if os.path.exists(local_path):

        with open(local_path, "rb") as f_in:
            with fs.open(s3_path, "wb") as f_out:
                f_out.write(f_in.read())

        print(f"Uploaded: {s3_path}")


In [ ]:
### Finished with training

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.evaluate()

In [ ]:
predictions = trainer.predict(test_dataset)

In [ ]:
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_true, y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

In [ ]:
y_pred_code = le.inverse_transform(y_pred)
y_true_code = le.inverse_transform(y_true)

In [ ]:
fs.ls("thierry57")

In [ ]:
# read file

BUCKET = "thierry57"
FILE_KEY_S3 = "naf_en_fr.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df_bpe = pd.read_csv(file_in, sep=";")

In [ ]:
#write file

BUCKET_OUT = "thierry57"
FILE_KEY_OUT_S3 = "nace_filtered.csv"
FILE_PATH_OUT_S3 = BUCKET_OUT + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    nace_filtered.to_csv(file_out)